In [ ]:
#@title 7. 实验总结

print(f"\\n{'='*70}")
print("持续学习实验总结")
print(f"{'='*70}")
print(f"数据集: {DATASET}")
print(f"Coreset方法: {CORESET_METHOD}")
print(f"每任务Coreset大小: {CORESET_SIZE_PER_TASK}")
print(f"训练轮数: {NUM_EPOCHS}")
print(f"任务数量: {len(tasks)}")
print(f"\\n主要结果:")
print(f"  平均准确率: {avg_acc:.2f}%")
print(f"  遗忘度: {forgetting:.2f}%")
print(f"  总Memory大小: {len(memory_labels)} 样本")
print(f"\\n各任务最终准确率:")
for task_id in range(len(tasks)):
    final_acc = results['task_accuracies'][-1][task_id]
    print(f"  任务 {task_id + 1}: {final_acc:.2f}%")
print(f"{'='*70}")

In [ ]:
#@title 6. 可视化结果

# 6.1 准确率矩阵
plt.figure(figsize=(10, 8))
accuracy_matrix = np.array(results['task_accuracies'])

# 创建热力图
im = plt.imshow(accuracy_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)

# 添加数值标签
for i in range(len(tasks)):
    for j in range(i + 1):
        text = plt.text(j, i, f'{accuracy_matrix[i, j]:.1f}',
                       ha="center", va="center", color="black", fontsize=10)

plt.colorbar(im, label='Accuracy (%)')
plt.xlabel('Task ID')
plt.ylabel('After Learning Task')
plt.title(f'Accuracy Matrix ({DATASET}, {CORESET_METHOD})')
plt.tight_layout()
plt.show()

# 6.2 任务准确率随时间变化
plt.figure(figsize=(12, 6))

for task_id in range(len(tasks)):
    # 获取任务task_id在学习过程中的准确率变化
    accuracies = []
    for learned_task in range(task_id, len(tasks)):
        accuracies.append(results['task_accuracies'][learned_task][task_id])
    
    plt.plot(range(task_id, len(tasks)), accuracies, marker='o', 
             label=f'Task {task_id + 1}', linewidth=2)

plt.xlabel('Tasks Learned')
plt.ylabel('Accuracy (%)')
plt.title(f'Task Accuracy Over Learning ({DATASET}, {CORESET_METHOD})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 6.3 Memory增长
memory_sizes = [CORESET_SIZE_PER_TASK * (i + 1) for i in range(len(tasks))]
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(tasks) + 1), memory_sizes, marker='o', linewidth=2, color='green')
plt.xlabel('Task ID')
plt.ylabel('Memory Size (samples)')
plt.title('Memory Growth Over Tasks')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title 5. 计算评估指标

def compute_average_accuracy(results):
    """计算平均准确率"""
    num_tasks = len(results['task_accuracies'])
    accuracies = []
    
    for task_id in range(num_tasks):
        # 获取学习完任务task_id后所有任务的准确率
        task_accs = results['task_accuracies'][task_id]
        accuracies.extend(task_accs)
    
    return np.mean(accuracies)

def compute_forgetting(results):
    """计算遗忘度"""
    num_tasks = len(results['task_accuracies'])
    forgetting = []
    
    for task_id in range(num_tasks - 1):  # 最后一个任务不会遗忘
        # 获取任务task_id在学习过程中的最大准确率
        max_acc = 0
        for learned_task in range(task_id + 1, num_tasks):
            acc = results['task_accuracies'][learned_task][task_id]
            max_acc = max(max_acc, acc)
        
        # 最终准确率
        final_acc = results['task_accuracies'][-1][task_id]
        
        # 遗忘度 = 最大准确率 - 最终准确率
        f = max_acc - final_acc
        forgetting.append(f)
    
    return np.mean(forgetting) if forgetting else 0.0

# 计算指标
avg_acc = compute_average_accuracy(results)
forgetting = compute_forgetting(results)

print("=" * 60)
print("评估指标")
print("=" * 60)
print(f"平均准确率: {avg_acc:.2f}%")
print(f"遗忘度: {forgetting:.2f}%")
print(f"Memory大小: {len(memory_labels)} 样本")
print("=" * 60)

In [ ]:
#@title 4.1 持续学习训练循环

from torch.utils.data import TensorDataset, ConcatDataset

# 存储所有任务的coreset
memory_coreset = []
memory_labels = []

# 存储性能结果
results = {
    'task_accuracies': [],  # 每个任务在所有测试集上的准确率
    'final_accuracies': []  # 每个任务结束时的准确率
}

# 训练循环
for task_id, task in enumerate(tasks):
    print(f"\\n{'='*60}")
    print(f"训练任务 {task_id + 1}/{len(tasks)}")
    print(f"{'='*60}")
    
    # 创建新模型
    model = SimpleCNN(
        in_channels=1 if 'MNIST' in DATASET else 3,
        num_classes=10
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # 准备当前任务的训练数据
    train_loader = DataLoader(task['train'], batch_size=BATCH_SIZE, shuffle=True)
    
    # 合并当前任务数据和memory中的coreset
    if len(memory_coreset) > 0:
        # 创建memory数据集
        memory_dataset = TensorDataset(
            torch.FloatTensor(memory_coreset),
            torch.LongTensor(memory_labels)
        )
        # 合并数据集
        combined_dataset = ConcatDataset([task['train'], memory_dataset])
        combined_loader = DataLoader(combined_dataset, batch_size=BATCH_SIZE, shuffle=True)
        print(f"当前任务样本: {len(task['train'])}, Memory样本: {len(memory_labels)}")
    else:
        combined_loader = train_loader
        print(f"当前任务样本: {len(task['train'])}, Memory样本: 0")
    
    # 训练
    model.train()
    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0.0
        pbar = tqdm(combined_loader, desc=f'Task {task_id+1}, Epoch {epoch+1}/{NUM_EPOCHS}')
        
        for X, y in pbar:
            X, y = X.to(device), y.to(device)
            
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        print(f'Epoch {epoch+1}: Loss={epoch_loss/len(combined_loader):.4f}')
    
    # 评估所有已见任务的性能
    task_accuracies = []
    for eval_task_id in range(task_id + 1):
        eval_task = tasks[eval_task_id]
        eval_loader = DataLoader(eval_task['test'], batch_size=BATCH_SIZE, shuffle=False)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in eval_loader:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                _, predicted = outputs.max(1)
                total += y.size(0)
                correct += predicted.eq(y).sum().item()
        
        acc = 100. * correct / total
        task_accuracies.append(acc)
        print(f"任务 {eval_task_id + 1} 准确率: {acc:.2f}%")
    
    results['task_accuracies'].append(task_accuracies)
    results['final_accuracies'].append(task_accuracies[-1])  # 当前任务的准确率
    
    # 从当前任务中选择coreset添加到memory
    print(f"\\n从任务 {task_id + 1} 中选择Coreset...")
    
    # 获取当前任务的所有数据
    all_data = []
    all_labels = []
    for X, y in DataLoader(task['train'], batch_size=len(task['train'])):
        all_data = X.view(X.size(0), -1).numpy()
        all_labels = y.numpy()
        break
    
    # 应用coreset方法
    method = get_coreset_method(CORESET_METHOD, CORESET_SIZE_PER_TASK)
    selected_indices, coreset_data, coreset_labels = method.select(all_data, all_labels)
    
    # 添加到memory
    memory_coreset.extend(coreset_data)
    memory_labels.extend(coreset_labels)
    
    print(f"选择了 {len(coreset_data)} 个样本")
    print(f"Memory总大小: {len(memory_labels)}")

print(f"\\n{'='*60}")
print("✓ 持续学习训练完成")
print(f"{'='*60}")

## 4. 持续学习训练与评估

使用Coreset方法进行经验回放，训练持续学习模型。

In [ ]:
#@title 3. 创建Split任务

def create_split_tasks(dataset_name='MNIST', num_tasks=5):
    """
    创建Split任务数据集
    
    Args:
        dataset_name: 数据集名称
        num_tasks: 任务数量
    
    Returns:
        tasks: 任务列表，每个任务包含(训练数据, 测试数据, 类别)
    """
    from torchvision import datasets, transforms
    
    # 数据预处理
    if dataset_name == 'MNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        full_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
        num_classes = 10
    else:  # CIFAR-10
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
        ])
        full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
        full_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
        num_classes = 10
    
    # 计算每个任务的类别
    classes_per_task = num_classes // num_tasks
    
    tasks = []
    for task_id in range(num_tasks):
        # 获取当前任务的类别范围
        start_class = task_id * classes_per_task
        end_class = start_class + classes_per_task if task_id < num_tasks - 1 else num_classes
        task_classes = list(range(start_class, end_class))
        
        # 筛选当前任务的训练数据
        train_indices = [i for i, (_, label) in enumerate(full_train) if label in task_classes]
        train_data = Subset(full_train, train_indices)
        
        # 筛选当前任务的测试数据
        test_indices = [i for i, (_, label) in enumerate(full_test) if label in task_classes]
        test_data = Subset(full_test, test_indices)
        
        tasks.append({
            'train': train_data,
            'test': test_data,
            'classes': task_classes
        })
        
        print(f"任务 {task_id + 1}: 类别 {task_classes}, 训练样本 {len(train_data)}, 测试样本 {len(test_data)}")
    
    return tasks

# 创建任务
print("正在创建Split任务...")
num_tasks = 5 if 'MNIST' in DATASET else 5
tasks = create_split_tasks('MNIST' if 'MNIST' in DATASET else 'CIFAR10', num_tasks=num_tasks)
print(f"\\n✓ 创建了 {len(tasks)} 个任务")

In [ ]:
#@title 2. 实验参数设置

#@markdown 数据集
DATASET = "Split MNIST"  #@param ["Split MNIST", "Split CIFAR-10"]

#@markdown Coreset方法
CORESET_METHOD = "Herding"  #@param ["Random", "K-Center", "Herding"]

#@markdown 每个任务的coreset大小
CORESET_SIZE_PER_TASK = 200  #@param {type:"integer"}

#@markdown 每个任务的训练轮数
NUM_EPOCHS = 5  #@param {type:"integer"}

#@markdown 批次大小
BATCH_SIZE = 128  #@param {type:"integer"}

device = get_device()

# 显示配置
print("=" * 60)
print("实验配置")
print("=" * 60)
print(f"数据集: {DATASET}")
print(f"Coreset方法: {CORESET_METHOD}")
print(f"每任务Coreset大小: {CORESET_SIZE_PER_TASK}")
print(f"训练轮数: {NUM_EPOCHS}")
print(f"批次大小: {BATCH_SIZE}")
print(f"设备: {device}")
print("=" * 60)

In [ ]:
#@title 1. 环境设置

import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import time
from pathlib import Path

# 设置随机种子
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 添加项目路径
def get_project_path():
    """自动检测项目路径"""
    if 'google.colab' in sys.modules:
        # Colab环境
        project_path = Path('/content/coreset_benchmark')
    else:
        # 本地环境
        project_path = Path().absolute().parent
    
    # 添加到sys.path
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))
    
    return project_path

project_root = get_project_path()

# 导入项目模块
from src.data import get_dataset
from src.coreset import get_coreset_method
from src.models import SimpleCNN
from src.utils import set_seed, get_device

print(f"✓ 环境设置完成")
print(f"  项目根目录: {project_root}")
print(f"  PyTorch版本: {torch.__version__}")
print(f"  CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU设备: {torch.cuda.get_device_name(0)}")

# 持续学习实验 (Continual Learning Experiment)

本notebook用于测试coreset方法在持续学习场景中的性能。

## 实验内容

1. **任务序列**: Split MNIST / Split CIFAR-10
2. **Coreset方法**:
   - Random Sampling (随机采样)
   - K-Center (K中心)
   - Herding (聚类)
3. **评估指标**:
   - 平均准确率 (Average Accuracy)
   - 遗忘度 (Forgetting Measure)
   - 内存使用

## 使用说明

1. 运行第一个单元格设置环境
2. 使用交互式控件调整参数
3. 运行实验并分析持续学习性能